In [1]:
# DAY 6: FINAL PIPELINE (IMPROVED)
# UCI Heart Disease Prediction
# PURPOSE:
#   - Load evaluation results from Day 5
#   - Load the best model and fitted preprocessor
#   - Save a final reusable bundle
#   - Keep all trained comparison models for transparency
#   - Generate final summary report and usage guide

import pandas as pd
import numpy as np
import joblib
import json
import os
import sys
from datetime import datetime

sys.path.append('scripts')
from utils import load_model, save_json


In [2]:
# SECTION 1: LOAD EVALUATION RESULTS & IDENTIFY BEST MODEL

print("LOADING EVALUATION RESULTS")

# Load evaluation report from Day 5
with open('outputs/reports/day5_evaluation_report.json', 'r') as f:
    evaluation_report = json.load(f)

best_model_name = evaluation_report['best_model']
best_model_metrics = evaluation_report['best_model_metrics']

print(f"✓ Best Model Identified: {best_model_name.upper()}")
print(f"\nBest Model Performance:")
for metric, value in best_model_metrics.items():
    print(f"  • {metric.replace('_', ' ').title():20s}: {value:.4f}")

LOADING EVALUATION RESULTS
✓ Best Model Identified: HIST_GRADIENT_BOOSTING

Best Model Performance:
  • Accuracy            : 0.6141
  • Precision           : 0.6072
  • Recall              : 0.6141
  • F1 Score            : 0.6096
  • Roc Auc             : 0.8293


In [3]:
# SECTION 2: LOAD BEST MODEL AND PREPROCESSOR

print("LOADING BEST MODEL AND PREPROCESSOR")

manifest_path = 'outputs/reports/day4_model_manifest.json'
with open(manifest_path, 'r') as f:
    model_manifest = json.load(f)

best_model_path = model_manifest[best_model_name]
best_model = load_model(best_model_path)
preprocessor = load_model('outputs/models/preprocessor.pkl')

print(f"✓ Loaded model: {best_model_name}")
print("✓ Loaded fitted preprocessor")


LOADING BEST MODEL AND PREPROCESSOR
✓ Loaded model: hist_gradient_boosting
✓ Loaded fitted preprocessor


In [4]:
# SECTION 3: REVIEW MODEL STORAGE

print("REVIEWING MODEL STORAGE")

model_dir = 'outputs/models'
current_model_files = list(model_manifest.values())

print(f"\nCurrent trained model files from manifest ({len(current_model_files)}):")
for model_path in current_model_files:
    print(f"  • {model_path}")

print("\n✓ Keeping all current trained model files for reproducibility and comparison")


REVIEWING MODEL STORAGE

Current trained model files from manifest (22):
  • outputs/models/dummy_baseline_trained.pkl
  • outputs/models/logistic_regression_trained.pkl
  • outputs/models/ridge_classifier_trained.pkl
  • outputs/models/sgd_classifier_trained.pkl
  • outputs/models/perceptron_trained.pkl
  • outputs/models/knn_trained.pkl
  • outputs/models/gaussian_nb_trained.pkl
  • outputs/models/linear_svm_trained.pkl
  • outputs/models/svm_rbf_trained.pkl
  • outputs/models/decision_tree_trained.pkl
  • outputs/models/random_forest_trained.pkl
  • outputs/models/extra_trees_trained.pkl
  • outputs/models/bagging_trained.pkl
  • outputs/models/adaboost_trained.pkl
  • outputs/models/gradient_boosting_trained.pkl
  • outputs/models/hist_gradient_boosting_trained.pkl
  • outputs/models/linear_discriminant_analysis_trained.pkl
  • outputs/models/nearest_centroid_trained.pkl
  • outputs/models/xgboost_trained.pkl
  • outputs/models/lightgbm_trained.pkl
  • outputs/models/catboost_train

In [5]:
# SECTION 4: SAVE FINAL BEST MODEL BUNDLE

print("SAVING FINAL BEST MODEL BUNDLE")

final_model_path = os.path.join(model_dir, f'{best_model_name}_final_bundle.pkl')
feature_names = preprocessor.get_feature_names_out().tolist()

final_bundle = {
    'model': best_model,
    'preprocessor': preprocessor,
    'feature_names': feature_names,
    'model_name': best_model_name,
    'target_column': 'num',
    'target_description': 'multiclass heart disease severity: 0=no disease, 1-4=increasing disease severity',
    'created_at': datetime.now().isoformat()
}

joblib.dump(final_bundle, final_model_path)

print(f"✓ Saved final bundle: {best_model_name}_final_bundle.pkl")


SAVING FINAL BEST MODEL BUNDLE
✓ Saved final bundle: hist_gradient_boosting_final_bundle.pkl


In [6]:
# SECTION 5: CREATE METADATA FILE FOR BEST MODEL

print("CREATING MODEL METADATA")

model_metadata = {
    'model_name': best_model_name,
    'model_version': '1.0',
    'creation_date': datetime.now().isoformat(),
    'performance_metrics': best_model_metrics,
    'model_type': 'Multiclass classifier',
    'dataset': 'UCI Heart Disease',
    'target': 'num: 0=no disease, 1-4=increasing disease severity',
    'file_path': final_model_path,
    'preprocessor_path': 'outputs/models/preprocessor.pkl',
    'final_artifact': 'model + fitted preprocessor bundle',
    'preprocessing': {
        'split_before_preprocessing': True,
        'numeric': 'SimpleImputer(median) + StandardScaler fitted on training data only',
        'categorical': 'SimpleImputer(most_frequent) + OneHotEncoder(handle_unknown=ignore) fitted on training data only'
    },
    'models_compared': len(evaluation_report['all_model_metrics'])
}
metadata_path = os.path.join(model_dir, f'{best_model_name}_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"✓ Saved metadata: {best_model_name}_metadata.json")


CREATING MODEL METADATA
✓ Saved metadata: hist_gradient_boosting_metadata.json


In [7]:
# SECTION 6: CREATE FINAL COMPREHENSIVE REPORT

print("GENERATING FINAL REPORT")

final_report = {
    'project_information': {
        'project_name': 'Heart Disease Prediction',
        'dataset': 'UCI Heart Disease',
        'generation_date': datetime.now().isoformat(),
        'pipeline_days': 6
    },

    'best_model': {
        'name': best_model_name,
        'version': '1.0',
        'type': 'Multiclass classifier with external fitted preprocessor',
        'metrics': best_model_metrics,
        'ranking': '1st place by weighted F1 score'
    },

    'all_models_evaluation': evaluation_report['all_model_metrics'],

    'model_ranking': {
        model: {
            'rank': rank + 1,
            'metrics': metrics
        }
        for rank, (model, metrics) in enumerate(
            sorted(evaluation_report['all_model_metrics'].items(),
                   key=lambda x: x[1]['f1_score'],
                   reverse=True)
        )
    },

    'recommendations': {
        'selected_model': best_model_name,
        'reason': f'Best weighted F1-score: {best_model_metrics["f1_score"]:.4f}',
        'next_steps': [
            'Review class-wise performance because rare severity classes are harder to predict',
            'Validate on external clinical data before real-world use',
            'Monitor model performance and retrain with new data',
            'Use the saved final bundle for reproducible inference'
        ]
    },

    'file_structure': {
        'final_model_bundle': final_model_path,
        'metadata_file': metadata_path,
        'comparison_table': 'outputs/reports/model_comparison_table.csv',
        'evaluation_report': 'outputs/reports/day5_evaluation_report.json',
        'preprocessor': 'outputs/models/preprocessor.pkl'
    }
}

report_path = 'outputs/reports/day6_final_pipeline_report.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"✓ Saved final report: {report_path}")


GENERATING FINAL REPORT
✓ Saved final report: outputs/reports/day6_final_pipeline_report.json


In [8]:
# SECTION 7: VISUAL SUMMARY

print("FINAL PIPELINE SUMMARY")

print(f"""

Model Name       : {best_model_name.upper():40}           
Accuracy         : {best_model_metrics['accuracy']:40.4f} 
Precision        : {best_model_metrics['precision']:40.4f}
Recall           : {best_model_metrics['recall']:40.4f}   
F1 Score         : {best_model_metrics['f1_score']:40.4f} 
ROC-AUC          : {best_model_metrics['roc_auc']:40.4f}  
Model Location   : outputs/models/                       
Report Location  : outputs/reports/                      
Config File      : {best_model_name}_metadata.json       
""")


FINAL PIPELINE SUMMARY


Model Name       : HIST_GRADIENT_BOOSTING                             
Accuracy         :                                   0.6141 
Precision        :                                   0.6072
Recall           :                                   0.6141   
F1 Score         :                                   0.6096 
ROC-AUC          :                                   0.8293  
Model Location   : outputs/models/                       
Report Location  : outputs/reports/                      
Config File      : hist_gradient_boosting_metadata.json       



In [9]:
# SECTION 8: CREATE SIMPLE USAGE GUIDE

print("HOW TO USE THE TRAINED MODEL")

usage_guide = f"""
# Loading and using the trained model bundle

import joblib
import pandas as pd

# Load the final bundle
bundle = joblib.load('outputs/models/{best_model_name}_final_bundle.pkl')
model = bundle['model']
preprocessor = bundle['preprocessor']
feature_names = bundle['feature_names']

# Prepare raw data with the same original feature columns used in training.
# Do not include id or num.
raw_rows = pd.DataFrame([...])

# Apply the saved training-fitted preprocessor, then predict.
X_new_array = preprocessor.transform(raw_rows)
X_new = pd.DataFrame(X_new_array, columns=feature_names)
predictions = model.predict(X_new)

if hasattr(model, 'predict_proba'):
    probabilities = model.predict_proba(X_new)

# Target interpretation:
# 0 = no heart disease
# 1-4 = increasing heart disease severity
"""

guide_path = 'outputs/reports/MODEL_USAGE_GUIDE.txt'
with open(guide_path, 'w') as f:
    f.write(usage_guide)

print(usage_guide)


HOW TO USE THE TRAINED MODEL

# Loading and using the trained model bundle

import joblib
import pandas as pd

# Load the final bundle
bundle = joblib.load('outputs/models/hist_gradient_boosting_final_bundle.pkl')
model = bundle['model']
preprocessor = bundle['preprocessor']
feature_names = bundle['feature_names']

# Prepare raw data with the same original feature columns used in training.
# Do not include id or num.
raw_rows = pd.DataFrame([...])

# Apply the saved training-fitted preprocessor, then predict.
X_new_array = preprocessor.transform(raw_rows)
X_new = pd.DataFrame(X_new_array, columns=feature_names)
predictions = model.predict(X_new)

if hasattr(model, 'predict_proba'):
    probabilities = model.predict_proba(X_new)

# Target interpretation:
# 0 = no heart disease
# 1-4 = increasing heart disease severity



In [10]:
# SECTION 9: FINAL STORAGE CHECK

print("FINAL STORAGE CHECK")

kept_files = list(model_manifest.values()) + [final_model_path, 'outputs/models/preprocessor.pkl']
print(f"\n✓ Current model artifacts kept ({len(kept_files)}):")
for f in kept_files:
    print(f"  • {f}")


FINAL STORAGE CHECK

✓ Current model artifacts kept (24):
  • outputs/models/dummy_baseline_trained.pkl
  • outputs/models/logistic_regression_trained.pkl
  • outputs/models/ridge_classifier_trained.pkl
  • outputs/models/sgd_classifier_trained.pkl
  • outputs/models/perceptron_trained.pkl
  • outputs/models/knn_trained.pkl
  • outputs/models/gaussian_nb_trained.pkl
  • outputs/models/linear_svm_trained.pkl
  • outputs/models/svm_rbf_trained.pkl
  • outputs/models/decision_tree_trained.pkl
  • outputs/models/random_forest_trained.pkl
  • outputs/models/extra_trees_trained.pkl
  • outputs/models/bagging_trained.pkl
  • outputs/models/adaboost_trained.pkl
  • outputs/models/gradient_boosting_trained.pkl
  • outputs/models/hist_gradient_boosting_trained.pkl
  • outputs/models/linear_discriminant_analysis_trained.pkl
  • outputs/models/nearest_centroid_trained.pkl
  • outputs/models/xgboost_trained.pkl
  • outputs/models/lightgbm_trained.pkl
  • outputs/models/catboost_trained.pkl
  • outp

In [11]:
# COMPLETION

print("READY")

print(f"""
Summary:
├─ Models Evaluated        : {len(evaluation_report['all_model_metrics'])}
├─ Best Model Identified   : {best_model_name}
├─ Best F1 Score           : {best_model_metrics['f1_score']:.4f}
├─ Final Bundle Saved      : {best_model_name}_final_bundle.pkl
├─ Metadata Generated      : {best_model_name}_metadata.json
├─ Report Generated        : day6_final_pipeline_report.json
└─ Comparison Models Kept  : Yes

Next Steps:
  1. Review the final report: outputs/reports/day6_final_pipeline_report.json
  2. Use the bundle: outputs/models/{best_model_name}_final_bundle.pkl
  3. Follow guide: outputs/reports/MODEL_USAGE_GUIDE.txt
""")


READY

Summary:
├─ Models Evaluated        : 22
├─ Best Model Identified   : hist_gradient_boosting
├─ Best F1 Score           : 0.6096
├─ Final Bundle Saved      : hist_gradient_boosting_final_bundle.pkl
├─ Metadata Generated      : hist_gradient_boosting_metadata.json
├─ Report Generated        : day6_final_pipeline_report.json
└─ Comparison Models Kept  : Yes

Next Steps:
  1. Review the final report: outputs/reports/day6_final_pipeline_report.json
  2. Use the bundle: outputs/models/hist_gradient_boosting_final_bundle.pkl
  3. Follow guide: outputs/reports/MODEL_USAGE_GUIDE.txt

